**Cell 1**

# NB12 — Train TinyLlama HelpSteer1 DPO Experts

This notebook trains five independent TinyLlama LoRA experts with Direct Preference Optimization (DPO), one per HelpSteer1 attribute. It retains exactly one pair per prompt. Both within each prompt and when reducing larger pools to 5,085 prompts, selection first maximizes the target-rating difference, then prefers the higher `chosen` target rating (for example, 4–2 before 3–1 or 2–0), and then minimizes differences on the other four attributes. Remaining exact ties are resolved with seed 137.

All experts use the same immutable base and dataset revisions, **5,085 unique prompts/pairs**, seed 137, optimizer, DPO beta, one fixed epoch, and LoRA architecture. Complexity has 5,085 prompts with at least one non-tied pair and therefore fixes the common one-pair-per-prompt budget. ArmoRM is absent from training and is used only for a strictly post-hoc differentiation check. `Run all` trains sequentially and stores each completed expert in a separate NB12 Google Drive directory.

**Cell 2**

## 1. Clone or update the repository

In [ ]:
# Cell 3
%cd /content
import os, shutil
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

**Cell 4**

## 2. Check the GPU

A bf16-capable CUDA GPU is required. Training loads TinyLlama and its PEFT reference path, but never ArmoRM.

In [ ]:
# Cell 5
!nvidia-smi

**Cell 6**

## 3. Install the pinned DPO runtime

TRL 0.11.4 is paired with Transformers 4.45.2 to avoid the incompatible `get_batch_samples` interface introduced in Transformers 4.46.

In [ ]:
# Cell 7
import importlib.metadata
import subprocess
import sys

print(f"Python {sys.version.split()[0]} ({sys.executable})")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
packages = [
    "pandas==2.2.3", "numpy==2.1.3", "protobuf==5.29.5",
    "transformers==4.45.2", "tokenizers==0.20.3",
    "peft==0.13.2", "accelerate==1.1.1", "trl==0.11.4",
    "datasets==3.1.0", "huggingface_hub==0.36.0",
    "bitsandbytes", "pyyaml", "safetensors",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "--only-binary=:all:", *packages],
    check=True,
)
expected = {"transformers": "4.45.2", "trl": "0.11.4"}
installed = {name: importlib.metadata.version(name) for name in expected}
assert installed == expected, (
    f"Incompatible DPO stack: {installed}; expected {expected}. "
    "Restart the Colab session and run the notebook from Cell 1."
)
import accelerate, peft, tokenizers, transformers, trl
print("Runtime:", sys.version.split()[0], "transformers", transformers.__version__,
      "tokenizers", tokenizers.__version__, "peft", peft.__version__,
      "accelerate", accelerate.__version__, "trl", trl.__version__)

**Cell 8**

## 4. Fixed settings and Drive persistence

The exact TinyLlama and HelpSteer1 revisions are pinned in this notebook and recorded in Drive. Existing matching experts are restored; incompatible cached runs fail closed.

In [ ]:
# Cell 9
from pathlib import Path
import json, os, shutil
from google.colab import drive

PROJECT_ROOT = Path("/content/master-thesis")
RUN_TAG = "nb12_helpsteer1_onepair5085_seed137_run1"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
BASE_REVISION = "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
DATASET_NAME = "nvidia/HelpSteer"
DATASET_REVISION = "3ca5d59c1bc1080af195b4254e7407db60b6f450"
ATTRIBUTES = ["helpfulness", "correctness", "coherence", "complexity", "verbosity"]
OUTPUT_ROOT = PROJECT_ROOT / "results" / "dpo_rq2" / RUN_TAG
DRIVE_ROOT = Path("/content/drive/MyDrive/master-thesis-nb12") / RUN_TAG
REVISION_FILE = DRIVE_ROOT / "source_revisions.json"
MAX_PAIRS = 5085
DPO_BETA = 0.1
EPOCHS = 1.0
LEARNING_RATE = 5e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256
SEED = 137
RUN_SMOKE_TEST = True
RUN_POSTHOC_EVALUATION = True
EVAL_PROMPTS = 64

drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
revisions = {
    "base_model": BASE_MODEL, "base_revision": BASE_REVISION,
    "dataset": DATASET_NAME, "dataset_revision": DATASET_REVISION,
    "pair_selection": "one_per_prompt_then_global_ranked", "max_pairs": MAX_PAIRS,
}
if REVISION_FILE.exists():
    assert json.loads(REVISION_FILE.read_text()) == revisions, "Drive revision binding differs."
else:
    REVISION_FILE.write_text(json.dumps(revisions, indent=2, sort_keys=True) + "\n")
for axis in ATTRIBUTES:
    source = DRIVE_ROOT / f"dpo_{axis}"
    target = OUTPUT_ROOT / f"dpo_{axis}"
    if source.exists() and not target.exists():
        shutil.copytree(source, target)
        print(f"[restore] {axis} from Drive")
print(f"output             = {OUTPUT_ROOT}")
print(f"Drive backup       = {DRIVE_ROOT}")
print(f"base revision      = {BASE_REVISION}")
print(f"dataset revision   = {DATASET_REVISION}")
print(f"budget             = {MAX_PAIRS} pairs per axis, {EPOCHS} fixed epoch")
print(f"effective batch    = {BATCH_SIZE * GRAD_ACCUM}")
print(f"post-hoc ArmoRM    = {RUN_POSTHOC_EVALUATION} (never use it for tuning)")

**Cell 10**

## 5. Validate all five HelpSteer1 one-pair-per-prompt pools

This loads no reward model and performs no training. Counts are bound to the pinned HelpSteer1 revision.

In [ ]:
# Cell 11
import importlib.util
from datasets import load_dataset

spec = importlib.util.spec_from_file_location(
    "nb12_dpo_script", PROJECT_ROOT / "scripts/0_dpo_expert_one_pair.py"
)
dpo_script = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(dpo_script)
assert tuple(ATTRIBUTES) == tuple(dpo_script.HELPSTEER_ATTRIBUTES)
train_rows = load_dataset(DATASET_NAME, split="train", revision=DATASET_REVISION)
assert len(train_rows) == 35331, f"Unexpected HelpSteer1 train size: {len(train_rows)}"
expected_candidates = {
    "helpfulness": 28021, "correctness": 27331, "coherence": 22734,
    "complexity": 16359, "verbosity": 23669,
}
expected_eligible_prompts = {
    "helpfulness": 7678, "correctness": 7562, "coherence": 6740,
    "complexity": 5085, "verbosity": 6748,
}
for axis in ATTRIBUTES:
    candidates = dpo_script.build_pairs_from_rows(train_rows, axis)
    assert len(candidates) == expected_candidates[axis], (axis, len(candidates))
    eligible_prompts = len({p['raw_prompt'] for p in candidates})
    assert eligible_prompts == expected_eligible_prompts[axis], (axis, eligible_prompts)
    selected = dpo_script.select_pairs(candidates, seed=SEED, max_pairs=MAX_PAIRS)
    unique_prompts = len({p['raw_prompt'] for p in selected})
    assert len(selected) == unique_prompts == MAX_PAIRS
    print(f"{axis:12} candidate_pairs={len(candidates):5d} "
          f"eligible_prompts={eligible_prompts:4d} selected={len(selected):4d}")
print("[OK] HelpSteer1 revision and equal-N=5,085 unique-prompt budget are valid.")

**Cell 12**

## 6. Evaluator-free DPO smoke test

A disposable 16-pair run checks the runtime, tokenizer, gradient checkpointing, and GPU before the five full jobs.

In [ ]:
# Cell 13
import subprocess, shutil, sys, os
from collections import deque

def run_logged_subprocess(command, label):
    print(f"[run] {label}: {' '.join(map(str, command))}", flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        command, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip())
    return_code = process.wait()
    if return_code:
        raise RuntimeError(
            f"{label} failed with exit code {return_code}.\n"
            f"Last subprocess output:\n{'\n'.join(tail)}"
        )

smoke_root = Path("/tmp/nb12_dpo_smoke")
all_axes_complete = all(
    (OUTPUT_ROOT / f"dpo_{axis}/training_manifest.json").is_file()
    and (OUTPUT_ROOT / f"dpo_{axis}/adapter/adapter_model.safetensors").is_file()
    for axis in ATTRIBUTES
)
if RUN_SMOKE_TEST and not all_axes_complete:
    shutil.rmtree(smoke_root, ignore_errors=True)
    command = [
        sys.executable, "-u", "scripts/0_dpo_expert_one_pair.py",
        "--reward_name", "helpfulness",
        "--base_model_name", BASE_MODEL, "--base_revision", BASE_REVISION,
        "--dataset_name", DATASET_NAME, "--dataset_revision", DATASET_REVISION,
        "--output_root", str(smoke_root), "--max_pairs", "16",
        "--seed", str(SEED), "--epochs", "1",
        "--save_steps", "1000", "--overwrite",
    ]
    run_logged_subprocess(command, "DPO smoke test")
    shutil.rmtree(smoke_root, ignore_errors=True)
    print("[OK] Smoke test passed and was deleted.")
elif all_axes_complete:
    print("[skip] All five verified adapters exist; smoke test is unnecessary.")
else:
    print("Smoke test disabled.")

**Cell 14**

## 7. Training helper

Every full call is identical except for `reward_name`. A completed expert is copied atomically to Drive. `--resume` uses a matching local trainer checkpoint if the current Colab runtime was interrupted without being destroyed.

In [ ]:
# Cell 15
import gc, torch

def train_axis(axis):
    assert axis in ATTRIBUTES
    command = [
        sys.executable, "-u", "scripts/0_dpo_expert_one_pair.py",
        "--reward_name", axis,
        "--base_model_name", BASE_MODEL, "--base_revision", BASE_REVISION,
        "--dataset_name", DATASET_NAME, "--dataset_revision", DATASET_REVISION,
        "--split", "train", "--output_root", str(OUTPUT_ROOT),
        "--beta", str(DPO_BETA), "--epochs", str(EPOCHS),
        "--lr", str(LEARNING_RATE), "--batch_size", str(BATCH_SIZE),
        "--grad_accum", str(GRAD_ACCUM), "--max_length", str(MAX_LENGTH),
        "--max_prompt_length", str(MAX_PROMPT_LENGTH),
        "--max_pairs", str(MAX_PAIRS), "--seed", str(SEED), "--resume",
    ]
    print(f"\n=== DPO {axis} ===")
    run_logged_subprocess(command, f"DPO training ({axis})")
    source = OUTPUT_ROOT / f"dpo_{axis}"
    destination = DRIVE_ROOT / f"dpo_{axis}"
    temporary = DRIVE_ROOT / f".dpo_{axis}.copying"
    shutil.rmtree(temporary, ignore_errors=True)
    shutil.copytree(source, temporary)
    shutil.rmtree(destination, ignore_errors=True)
    temporary.rename(destination)
    print(f"[backup] {axis} -> {destination}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

**Cell 16**

### Helpfulness expert

In [ ]:
# Cell 17
train_axis("helpfulness")

**Cell 18**

### Correctness expert

In [ ]:
# Cell 19
train_axis("correctness")

**Cell 20**

### Coherence expert

In [ ]:
# Cell 21
train_axis("coherence")

**Cell 22**

### Complexity expert

In [ ]:
# Cell 23
train_axis("complexity")

**Cell 24**

### Verbosity expert

In [ ]:
# Cell 25
train_axis("verbosity")

**Cell 26**

## 8. Verify and bundle all five adapters

The compact ZIP contains adapter files and immutable manifests but excludes resumable trainer checkpoints.

In [ ]:
# Cell 27
import hashlib
bundle_stage = Path("/tmp/nb12_dpo_bundle")
shutil.rmtree(bundle_stage, ignore_errors=True)
bundle_stage.mkdir(parents=True)
shared = {}
shared_keys = [
    "trainer_script_sha256", "runtime_versions", "base_model_name",
    "base_revision", "dataset_name", "dataset_revision",
    "pair_rule", "one_pair_per_prompt", "selected_pair_count",
    "selected_unique_prompt_count", "seed", "beta", "dpo_disable_dropout",
    "epochs", "learning_rate", "effective_batch_size", "lora",
]
for axis in ATTRIBUTES:
    expert = OUTPUT_ROOT / f"dpo_{axis}"
    manifest_path = expert / "training_manifest.json"
    weights_path = expert / "adapter/adapter_model.safetensors"
    assert manifest_path.is_file() and weights_path.is_file(), f"Incomplete expert: {axis}"
    manifest = json.loads(manifest_path.read_text())
    assert manifest["completed"] and manifest["armorm_used_during_training"] is False
    assert manifest["selected_pair_count"] == MAX_PAIRS and manifest["seed"] == SEED
    assert manifest["dataset_name"] == DATASET_NAME
    assert manifest["dataset_revision"] == DATASET_REVISION
    digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
    assert digest == manifest["adapter_model_sha256"]
    for key in shared_keys:
        value = json.dumps(manifest[key], sort_keys=True)
        if key in shared:
            assert shared[key] == value, f"Experts differ on {key}"
        shared[key] = value
    target = bundle_stage / f"dpo_{axis}"
    shutil.copytree(expert / "adapter", target / "adapter")
    shutil.copy2(manifest_path, target / "training_manifest.json")
    print(f"[OK] {axis:12} {digest}")
bundle_base = Path("/content/nb12_tinyllama_helpsteer1_dpo_onepair5085_adapters")
bundle_path = Path(shutil.make_archive(str(bundle_base), "zip", root_dir=bundle_stage))
bundle_sha = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
sha_path = Path(str(bundle_path) + ".sha256")
sha_path.write_text(f"{bundle_sha}  {bundle_path.name}\n")
shutil.copy2(bundle_path, DRIVE_ROOT / bundle_path.name)
shutil.copy2(sha_path, DRIVE_ROOT / sha_path.name)
print(f"bundle = {bundle_path} ({bundle_path.stat().st_size/1e6:.1f} MB)")
print(f"sha256 = {bundle_sha}")

**Cell 28**

## 9. Post-hoc expert differentiation

After all fixed adapters are complete, the untouched base and five experts generate responses for the same 64 HelpSteer1 validation prompts. ArmoRM scores the generations on all five axes. This diagnostic must not be used for tuning, and its prompts must be excluded from any later confirmatory evaluation.

In [ ]:
# Cell 29
EVAL_DIR = OUTPUT_ROOT / "posthoc_differentiation"
destination = DRIVE_ROOT / "posthoc_differentiation"
if RUN_POSTHOC_EVALUATION:
    if destination.exists() and not EVAL_DIR.exists():
        shutil.copytree(destination, EVAL_DIR)
        print(f"[restore] evaluation from {destination}")
    command = [
        sys.executable, "-u", "scripts/diff_experts.py",
        "--base_model_name", BASE_MODEL, "--base_revision", BASE_REVISION,
        "--expert_root", str(OUTPUT_ROOT), "--adapter_subdir", "adapter",
        "--dataset_name", DATASET_NAME, "--dataset_revision", DATASET_REVISION,
        "--split", "validation", "--n_prompts", str(EVAL_PROMPTS),
        "--prompt_seed", "991", "--max_new_tokens", "128",
        "--armorm_precision", "8bit", "--output_dir", str(EVAL_DIR),
    ]
    run_logged_subprocess(command, "post-hoc expert differentiation")
    shutil.rmtree(destination, ignore_errors=True)
    shutil.copytree(EVAL_DIR, destination)
    print(f"[backup] evaluation -> {destination}")
else:
    print("Post-hoc evaluation disabled; the five DPO adapters are still complete.")

**Cell 30**

## 10. Read the differentiation result

In [ ]:
# Cell 31
import pandas as pd
report_path = EVAL_DIR / "dpo_expert_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    matrix = pd.read_csv(EVAL_DIR / "dpo_expert_payoff_matrix.csv", index_col=0)
    display(matrix.round(4))
    print(f"Own expert leads its ArmoRM column: {report['n_own_column_leaders']}/{len(ATTRIBUTES)}")
    print("Own-axis change relative to untouched base:")
    for axis, value in report["own_axis_minus_base"].items():
        print(f"  {axis:12} {value:+.4f}")
    print(f"Diagnostic prompt SHA256: {report['prompt_sha256']}")
else:
    print("No post-hoc report because Cell 29 was disabled or has not run.")

**Cell 32**

## 11. Download the adapter bundle

In [ ]:
# Cell 33
from google.colab import files
assert bundle_path.exists() and sha_path.exists()
files.download(str(bundle_path))
files.download(str(sha_path))